# Çözücüler ⚙️

Bu egzersizde, farklı `çözücülerin` `LogisticRegression` modelleri üzerindeki etkilerini araştıracaksınız.

👇 Veri kümesini içe aktarmak için aşağıdaki kodu çalıştırın

In [1]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


- Veri kümesi farklı şaraplardan oluşmaktadır 🍷
- Özellikler şarapların farklı niteliklerini tanımlar 
- Hedef 🎯 bir uzman tarafından verilen kalite değerlendirmesidir

## 1. Hedef mühendisliği

Bu bölümde, değerlendirmeleri ikili bir hedefe dönüştüreceksiniz.

👇 Her değerlendirme için kaç gözlem bulunmaktadır?

In [5]:
import pandas as pd
import numpy as np

print(df["quality rating"].value_counts())
print(df["quality rating"].value_counts(normalize=True))

quality rating
10    10143
5     10124
1     10090
2     10030
8      9977
6      9961
9      9955
7      9954
4      9928
3      9838
Name: count, dtype: int64
quality rating
10    0.10143
5     0.10124
1     0.10090
2     0.10030
8     0.09977
6     0.09961
9     0.09955
7     0.09954
4     0.09928
3     0.09838
Name: proportion, dtype: float64


❓ Hedefi ikili sınıflandırma görevine dönüştürerek `y` oluşturun, burada 6'nın altındaki kalite değerlendirmeleri kötü [0], 6 ve üzeri değerlendirmeler iyi [1] olacak

In [7]:
import numpy as np

# 6'dan küçükse 0, değilse 1 yap
df["target"] = np.where(df["quality rating"] < 6, 0, 1)


❓ Yeni ikili hedefin sınıf dengesini kontrol edin

In [8]:
df["target"].value_counts()

target
0    50010
1    49990
Name: count, dtype: int64

❓ Özellikleri normalleştirerek `X`'inizi oluşturun. Bu farklı çözücülerin adil karşılaştırılmasına olanak sağlayacaktır.

In [12]:
from sklearn.preprocessing import StandardScaler
X = df[["fixed acidity","volatile acidity","citric acid","residual sugar","chlorides","free sulfur dioxide","total sulfur dioxide","density","sulphates","alcohol"]]
y= df["target"]
scaler = StandardScaler()
X_rescaled = scaler.fit_transform(X)
X_rescaled = pd.DataFrame(X_rescaled, columns=X.columns)

In [13]:
X_rescaled

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,-0.788603,-1.528461,-1.733180,0.461130,-1.526653,-0.852381,-0.221393,-0.478387,0.340231,-0.489833
1,-0.346860,-0.462069,-0.158290,-0.783868,0.117066,-3.102634,-0.301357,-0.986972,-0.769429,-0.489833
2,0.064417,0.232757,0.550411,1.017553,-0.439117,0.423432,1.457850,1.246811,1.339925,-0.307387
3,0.376684,-0.610695,-1.103224,0.099454,-1.015163,0.925720,-3.030126,1.067310,0.660133,-0.426679
4,1.229704,1.247129,0.228871,0.308113,0.583862,-0.621328,0.218409,-0.957055,-2.318954,1.320591
...,...,...,...,...,...,...,...,...,...,...
99995,-2.723130,-2.078377,-1.149158,-0.630851,-0.250412,1.076407,-1.620762,0.887810,2.419594,-0.356507
99996,0.049185,-0.194543,-0.112355,-0.366550,-0.071639,0.041693,0.868116,1.276728,-0.429533,-0.370541
99997,-0.209768,0.333079,1.140995,1.567021,-0.518571,-0.972930,-0.071460,-0.139331,0.040323,-0.588073
99998,-2.479410,-2.279022,-1.949728,-0.422192,-0.707276,-0.249635,1.447854,0.209697,-1.679150,-0.040736


## 2. LogisticRegression çözücüleri

❓ Lojistik Regresyon modelleri farklı **çözücüler** kullanılarak optimize edilebilir. Mevcut çözücülerin karşılaştırmasını yapın:
- Uyum süresi - hangi çözücü **en hızlı**?
- Kesinlik - kesinlik puanları **ne kadar farklı**?

Lojistik Regresyon için mevcut çözücüler: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
Bu 5 çözücü hakkında daha fazla bilgi için [bu Stack Overflow konusuna](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions) göz atın

In [20]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score 

# 1. Veri Hazırlığı yapalım
X = df[["fixed acidity","volatile acidity","citric acid","residual sugar",
        "chlorides","free sulfur dioxide","total sulfur dioxide",
        "density","sulphates","alcohol"]]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Solver Karşılaştırma Döngüsü
solvers = ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']
results = []

print(f"{'Solver':<12} | {'Süre (sn)':<10} | {'Precision':<10}")
print("-" * 38)

for s in solvers:
    start_time = time.time()
    
    # Modeli kur ve eğit
    model = LogisticRegression(solver=s, max_iter=1000)
    model.fit(X_train_scaled, y_train)
    
    end_time = time.time()
    
    # Tahmin ve Precision Skorlama
    y_pred = model.predict(X_test_scaled)
    # Binary sınıflandırma olduğu için doğrudan hesaplar
    prec = precision_score(y_test, y_pred) 
    
    elapsed = end_time - start_time
    
    results.append({'Solver': s, 'Time': elapsed, 'Precision': prec})
    print(f"{s:<12} | {elapsed:<10.4f} | {prec:<10.4f}")

# 3. Sonucu Yorumlama
best_prec_solver = max(results, key=lambda x: x['Precision'])
fastest_solver = min(results, key=lambda x: x['Time'])

print(f"\nEn Yüksek Precision: {best_prec_solver['Solver']} ({best_prec_solver['Precision']:.4f})")
print(f"En Hızlı Çözücü: {fastest_solver['Solver']} ({fastest_solver['Time']:.4f} sn)")

Solver       | Süre (sn)  | Precision 
--------------------------------------
lbfgs        | 0.0939     | 0.8801    
liblinear    | 0.1394     | 0.8801    
newton-cg    | 0.1010     | 0.8801    
sag          | 0.7470     | 0.8801    
saga         | 1.3661     | 0.8801    

En Yüksek Precision: lbfgs (0.8801)
En Hızlı Çözücü: lbfgs (0.0939 sn)


In [17]:
fastest_solver = "lbfgs"

<details>
    <summary>ℹ️ Yorumumuz için buraya tıklayın</summary>

Maliyet fonksiyonumuz 5 çözücünün de bulduğu global bir minimuma sahip olacak kadar "kolay" olduğundan, tüm çözücüler benzer kesinlik puanları üretmelidir. Derin Öğrenme'de olduğu gibi çok karmaşık maliyet fonksiyonları için, farklı çözücüler kayıp fonksiyonunun farklı değerlerinde durabilir.

**Şarap veri kümesi**
    
Mevcut veri kümesinde sklearn'in <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> ile özellik önemini kontrol ederseniz, birçok özelliğin neredeyse 0 önemine sahip olduğunu göreceksiniz. Liblinear çözücü, bir defada sadece *bir* yön boyunca hareket eder ve diğerlerini L1 düzenlileştirme ile düzenler (yani, beta değerlerini 0'a ayarlar), bu da birçok özelliğin hedefi tahmin etmede o kadar da önemli olmadığı bir veri kümesi için iyi bir uyum sağlayabilir.

❗️En iyi çözücüyü arama maliyeti vardır. Varsayılanla (`lbfgs`) devam etmek genel olarak en çok zaman tasarrufu sağlayabilir, sklearn başlamak için hangi çözücüyü seçeceğiniz konusunda fikir vermek için bu tabloyu sunar: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

###  🧪 Kodunuzu test edin

In [18]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D4-S-data-solvers/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stokastik Gradyan İnişi

Lojistik Regresyon modelleri ayrıca Stokastik Gradyan İnişi ile de optimize edilebilir.

❓ **Stokastik Gradyan İnişi** ile optimize edilmiş bir Lojistik Regresyon modelini değerlendirin. Kesinlik puanı ve eğitim süresi 2. bölümde eğitilen modellerin performansı ile nasıl karşılaştırılır?

<details>
<summary>💡 İpucu</summary>

- Takılırsanız, [SGDClassifier belgelerine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) bakın!

</details>

In [21]:
from sklearn.linear_model import SGDClassifier

# SGD Modeli (loss='log_loss' lojistik regresyon yapar)
sgd_model = SGDClassifier(loss='log_loss', max_iter=1000, random_state=42)

start_time = time.time()
sgd_model.fit(X_train_scaled, y_train)
end_time = time.time()

# Precision hesaplama
y_pred_sgd = sgd_model.predict(X_test_scaled)
prec_sgd = precision_score(y_test, y_pred_sgd)

print(f"SGD | Süre: {end_time - start_time:.4f} sn | Precision: {prec_sgd:.4f}")

SGD | Süre: 0.1778 sn | Precision: 0.8650


☝️ SGD modeli, benzer performans için en kısa sürelerden birine sahip olmalıdır (hatta `liblinear`'dan bile daha kısa olabilir). Bu, Gradyan İnişinin her dönemini aynı anda 100k satırı belleğe yüklemek yerine tek bir satırda gerçekleştirmenin doğrudan bir etkisidir.

<h2>En Hızlı Yakınsayan Çözücü: lbfgs</h2>

<h3>Gerekçe:</h3> <b>Veri setimizdeki öznitelik (feature) sayısının nispeten az olması ve verilerin önceden ölçeklendirilmiş olması nedeniyle lbfgs algoritması en verimli performansı göstermiştir. İkinci dereceden türev bilgilerini (Hessian approximation) kullanarak hedefe doğru daha büyük adımlar atması ve her iterasyonda matematiksel olarak daha kararlı bir yol izlemesi, stokastik yöntemlere (SAG/SAGA) kıyasla daha az adımda yakınsamasını sağlamıştır.</b>

## 4. Tahminler

❓ En iyi modeli (kısa uyum süresi ve yüksek kesinlik ile dengelenen) kullanarak aşağıdaki şarabın ikili kalitesini (0 veya 1) tahmin edin. Şunları kaydedin:
- `predicted_class`
- `predicted_proba_of_class` (yani modeliniz 1 sınıfını tahmin ettiyse, 1'in sınıf olması gerektiğine inanma olasılığı nedir, 0 ile 1 arasında olmalıdır)

In [37]:
new_wine = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv')
new_wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.54,13.5,12.35,8.78,14.72,9.06,9.67,10.15,11.17,12.17


In [38]:
# --- 1. Veriyi Hazırla ve Ölçeklendir ---
# Sadece eğitim verisiyle fit et, sonra her ikisini transform et
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 2. Modeli En İyi Ayarlarla Eğit ---
best_model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
best_model.fit(X_train_scaled, y_train)

# --- 3. Yeni Şarabı Tahmin Et ---
new_wine_scaled = scaler.transform(new_wine) 

# --- 4. Tahmin ve Olasılık Kaydı ---
predicted_class = best_model.predict(new_wine_scaled)[0]

# Olasılıkları al
probs = best_model.predict_proba(new_wine_scaled)[0]


predicted_proba_of_class = probs[predicted_class]

print(f"Predicted Class: {predicted_class}")
print(f"Probability: {predicted_proba_of_class}")

Predicted Class: 0
Probability: 0.9689747269625119


# 🏁  Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [39]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D4-S-data-solvers/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 2 items

test_new_data_prediction.py::TestNewDataPrediction::test_predicted_class PASSED [ 50%]
test_new_data_prediction.py::TestNewDataPrediction::test_predicted_proba PASSED [100%]

============================== 2 passed in 0.21s ===============================


💯 You can commit your code:

git add tests/new_data_prediction.pickle

git commit -m 'Completed new_data_prediction step'

git push origin master

